# Chapitre 5 - Cours 2 : splines de lissage, degrés de liberté et choix de $\lambda$

Les splines de régression du cours 1 obligent à choisir le nombre et la position des
nœuds. Les **splines de lissage** contournent ce problème : on met un nœud en **chaque**
observation et l'on contrôle la complexité par une **pénalité** de courbure. Plan :

1. le critère pénalisé et sa solution (§5.4) ;
2. calcul matriciel en base de B-splines, données *bone* (figure 5.6) ;
3. matrice de lissage, degrés de liberté effectifs, forme de Reinsch et valeurs
   propres (§5.4.1, figures 5.7 et 5.8) ;
4. choix de $\lambda$ : compromis biais-variance et validation croisée (§5.5, figure 5.9) ;
5. régression logistique non paramétrique (§5.6).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl.ch05_splines import (SmoothingSpline, bspline_basis, bspline_penalty_matrix,
                              natural_spline_basis, loocv_score, gcv_score)
set_style()

## 1. Le critère des splines de lissage

Parmi toutes les fonctions $f$ deux fois dérivables, on cherche celle qui minimise la
**somme des carrés résiduels pénalisée** (éq. (5.9) du livre)

$$
\mathrm{RSS}(f, \lambda) = \sum_{i=1}^{N}\{y_i - f(x_i)\}^2
  + \lambda \int \{f''(t)\}^2\,dt .
$$

Le premier terme mesure l'**attache aux données**, le second la **courbure** de $f$
($f''$ est nulle pour une droite). Le paramètre $\lambda \ge 0$ règle le compromis :

- $\lambda = 0$ : aucune pénalité, $f$ peut être n'importe quelle fonction interpolant
  les données ;
- $\lambda = \infty$ : aucune courbure tolérée, $f$ est la droite des moindres carrés.

**Théorème (exercice 5.7).** Le problème, posé sur un espace de fonctions de dimension
infinie, admet une solution **unique**, qui est une **spline cubique naturelle** dont les
nœuds sont les valeurs distinctes des $x_i$.

> **Intuition.** Fixons les valeurs $f(x_i) = z_i$. Le premier terme ne dépend plus que
> des $z_i$ ; il reste à minimiser $\int f''^2$ parmi les fonctions qui passent par les
> points $(x_i, z_i)$ : c'est la spline cubique naturelle interpolante qui réalise ce
> minimum (exercice 5.7). On peut donc se restreindre à cet espace de dimension $N$.